# M0 Baseline：先让邮件 Agent 失忆

本 Notebook 只运行无记忆基线，记录路由错误和写作偏好错误。不要先加向量库；先确认失败发生在哪一层。

全程离线，不需要 API Key。

In [ ]:
import json

from email_memory_lab import (
    AgentConfig,
    LocalStore,
    evaluate_config,
    find_project_root,
    load_labeled_emails,
    seed_memories,
    start_run,
)

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
cases = load_labeled_emails(DATA / "emails.json")
store = LocalStore()
seed_memories(store, DATA / "memories.json", "user-a")
run = start_run(PROJECT_ROOT, "01-baseline", {"memory": "none"})

print(f"Project: {PROJECT_ROOT}")
print(f"Run: {run.run_id}")

## 运行无记忆基线

记忆已经写入 `store`，但 baseline 配置不会读取它。这样才能区分“没有写入”和“写入了但没有使用”。

In [ ]:
baseline = evaluate_config(cases, store, "user-a", AgentConfig())
print(json.dumps(baseline["summary"], ensure_ascii=False, indent=2))

In [ ]:
for row in baseline["failures"]:
    print(
        f"{row['email_id']}: "
        f"expected={row['expected_action']} predicted={row['predicted_action']} "
        f"writing_ok={row['writing_ok']} memory={row['used_memory']}"
    )

## 观察问题

- `vendor-pricing` 和 `event-invite` 被错误地归为 respond；
- 所有应回复邮件都缺少用户签名；
- 必需记忆召回为 0，因为 baseline 没有读取三类记忆。

下一步只增加 semantic memory，观察写作偏好是否改善。

In [ ]:
path = run.save("baseline-results.json", baseline)
print(f"Saved: {path}")